# Guided BERTopic notebook with nine major topics


1. **Intertopic distance map**
2. **Hierarchical clustering dendrogram**
3. **t-SNE document map (static Matplotlib version matching the requested style)**

With a **safe CSV reader** so UTF-8 / UTF-8-BOM / Arabic text does not turn into mojibake like `ø¹ø...`.


In [ ]:

# !pip install bertopic sentence-transformers umap-learn hdbscan scikit-learn pandas numpy plotly openpyxl kaleido


In [ ]:
from pathlib import Path

# =========================
# Configuration
# =========================
INPUT_CSV = "all.csv"   # change this
OUTPUT_DIR = "bertopic_all_visuals_9_topics_output"

TEXT_COL = "content"
TITLE_COL = "title"
DATE_COL = "date"
ADDRESS_COL = "address"
CATEGORY_COL = "Categories"

SEED = 1234
# Use a multilingual sentence-transformer to support Arabic/Chinese/English mixed corpora
EMBEDDING_MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"
TOP_N_WORDS = 10
SEED_MULTIPLIER = 2.0

# Keep exactly 9 major topics after fitting
TARGET_MAJOR_TOPICS = 9

# HDBSCAN/UMAP settings
UMAP_N_NEIGHBORS = 15
UMAP_N_COMPONENTS = 5
UMAP_MIN_DIST = 0.0
MIN_CLUSTER_SIZE = 15
MIN_SAMPLES = 3
NGRAM_RANGE = (1, 2)
MIN_DF = 2

# t-SNE settings for document map
TSNE_PERPLEXITY = 30
TSNE_LEARNING_RATE = 200.0
TSNE_MAX_ITER = 1000

# Optional: assign outliers (-1) back into nearest topics after reduction.
REDUCE_OUTLIERS = False
OUTLIER_STRATEGY = "c-tf-idf"

# Guided topics from your table
SEED_TOPICS = {
    "Economic Cooperation": [
        "trade", "investment", "economic", "partnership", "oil", "infrastructure"
    ],
    "Belt and Road": [
        "belt and road", "bri", "connectivity", "silk road", "corridor", "port"
    ],
    "Diplomatic Relations": [
        "diplomatic", "embassy", "minister", "bilateral", "cooperation", "visit"
    ],
    "Development and Aid": [
        "development", "aid", "support", "capacity", "training", "assistance"
    ],
    "Security and Stability": [
        "security", "stability", "defense", "terrorism", "peace", "conflict"
    ],
    "Culture and Education": [
        "culture", "education", "scholarship", "students", "university", "exchange"
    ],
    "Energy and Resources": [
        "energy", "oil", "gas", "renewable", "electricity", "resources"
    ],
    "Health and Social Affairs": [
        "health", "medical", "hospital", "social", "women", "youth"
    ],
    "Technology and Innovation": [
        "technology", "digital", "innovation", "ai", "research", "startup"
    ],
}


In [ ]:
import html
import re
import warnings
import numpy as np
import pandas as pd
import plotly.io as pio
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from hdbscan import HDBSCAN
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.manifold import TSNE
from umap import UMAP

warnings.filterwarnings("ignore")


MOJIBAKE_MARKERS = (
    "Ã", "Â", "Ð", "Ñ", "Ø", "Ù", "ø", "ð", "þ", "�", "œ", "ž", "â€", "â€™", "â€œ", "â€"
)


def safe_slug(value):
    value = value if isinstance(value, str) and value.strip() else "missing"
    value = re.sub(r"[^A-Za-z0-9]+", "_", value)
    value = re.sub(r"_+", "_", value).strip("_")
    return value.lower() or "missing"


def normalize_spaces(text):
    text = "" if text is None else str(text)
    text = text.replace("\r", " ").replace("\n", " ").replace("\t", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def clean_html_text(text):
    text = "" if text is None else str(text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = html.unescape(text)
    return normalize_spaces(text)


def mojibake_score(text):
    text = "" if text is None else str(text)
    score = 0
    for marker in MOJIBAKE_MARKERS:
        score += text.count(marker)
    return score


def maybe_fix_mojibake(text):
    text = "" if text is None else str(text)
    base_score = mojibake_score(text)
    best = text
    best_score = base_score

    # Try common "decoded with the wrong single-byte codec" repairs
    for enc in ("latin-1", "cp1252"):
        try:
            repaired = text.encode(enc, errors="ignore").decode("utf-8", errors="ignore")
            repaired = normalize_spaces(repaired)
            score = mojibake_score(repaired)
            if repaired and score < best_score:
                best = repaired
                best_score = score
        except Exception:
            pass
    return best


def count_non_ascii(text):
    return sum(ord(ch) > 127 for ch in str(text))


def frame_garble_score(df):
    sample_text = " ".join(
        df.astype(str).head(200).fillna("").agg(" ".join, axis=1).tolist()
    )
    return mojibake_score(sample_text), count_non_ascii(sample_text)


def read_csv_safely(path):
    # Do not start with latin-1; it often "succeeds" while silently garbling UTF-8 text.
    encodings_to_try = ["utf-8-sig", "utf-8", "utf-16", "cp1256", "cp1252", "latin-1"]
    candidates = []
    last_error = None

    for enc in encodings_to_try:
        try:
            tmp = pd.read_csv(path, encoding=enc)
            garble, non_ascii = frame_garble_score(tmp)
            candidates.append((garble, -non_ascii, enc, tmp))
        except Exception as e:
            last_error = e

    if not candidates:
        raise last_error if last_error is not None else ValueError("Could not read the CSV.")

    candidates.sort(key=lambda x: (x[0], x[1]))
    best_garble, _, best_encoding, best_df = candidates[0]
    print(f"Selected CSV encoding: {best_encoding} | garble score={best_garble}")
    return best_df, best_encoding


def ensure_columns(df):
    if TEXT_COL not in df.columns:
        raise ValueError(f"Your CSV must contain a '{TEXT_COL}' column.")
    df = df.copy()
    for col in [TITLE_COL, DATE_COL, ADDRESS_COL, CATEGORY_COL]:
        if col not in df.columns:
            df[col] = ""
    for col in [TEXT_COL, TITLE_COL, DATE_COL, ADDRESS_COL, CATEGORY_COL]:
        df[col] = df[col].fillna("").astype(str)
        df[col] = df[col].map(clean_html_text).map(maybe_fix_mojibake)
    return df


def prepare_input(df):
    df = ensure_columns(df)
    prepared = df.copy()
    prepared["text_for_bertopic"] = prepared[TEXT_COL].map(normalize_spaces)
    prepared["hover_text"] = np.where(
        prepared[TITLE_COL].str.strip().ne(""),
        prepared[TITLE_COL].map(normalize_spaces),
        prepared[TEXT_COL].map(normalize_spaces),
    )
    prepared = prepared.loc[prepared["text_for_bertopic"].str.strip().ne("")].reset_index(drop=True)
    prepared["doc_id"] = np.arange(1, len(prepared) + 1)
    return prepared


def save_csv(df_obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df_obj.to_csv(path, index=False, encoding="utf-8-sig")


def try_save_plotly(fig, html_path, png_path=None):
    html_path = Path(html_path)
    html_path.parent.mkdir(parents=True, exist_ok=True)
    fig.write_html(str(html_path))
    if png_path is not None:
        try:
            fig.write_image(str(png_path), scale=2)
        except Exception as e:
            print(f"Could not write PNG {png_path}: {e}")


def build_topic_model():
    embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

    umap_model = UMAP(
        n_neighbors=UMAP_N_NEIGHBORS,
        n_components=UMAP_N_COMPONENTS,
        min_dist=UMAP_MIN_DIST,
        metric="cosine",
        random_state=SEED,
    )

    hdbscan_model = HDBSCAN(
        min_cluster_size=MIN_CLUSTER_SIZE,
        min_samples=MIN_SAMPLES,
        metric="euclidean",
        cluster_selection_method="eom",
        prediction_data=True,
    )

    vectorizer_model = CountVectorizer(
        stop_words=None,
        ngram_range=NGRAM_RANGE,
        min_df=MIN_DF,
        token_pattern=r"(?u)\b\w\w+\b",
    )

    seed_words_flat = sorted({w for words in SEED_TOPICS.values() for w in words})
    ctfidf_model = ClassTfidfTransformer(
        seed_words=seed_words_flat,
        seed_multiplier=SEED_MULTIPLIER,
    )

    topic_model = BERTopic(
        embedding_model=embedding_model,
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer_model,
        ctfidf_model=ctfidf_model,
        seed_topic_list=list(SEED_TOPICS.values()),
        top_n_words=TOP_N_WORDS,
        calculate_probabilities=True,
        verbose=True,
    )
    return topic_model, embedding_model


def get_topic_terms(topic_model, topic_ids):
    rows = []
    for topic_id in topic_ids:
        topic_terms = topic_model.get_topic(topic_id)
        if not topic_terms:
            continue
        for rank, pair in enumerate(topic_terms, start=1):
            if isinstance(pair, (list, tuple)) and len(pair) >= 2:
                term, weight = pair[0], pair[1]
            else:
                continue
            rows.append(
                {
                    "Topic": topic_id,
                    "Rank": rank,
                    "Term": str(term),
                    "Weight": float(weight),
                }
            )
    return pd.DataFrame(rows)


def tsne_2d(embeddings):
    if embeddings is None or len(embeddings) < 3:
        return None
    perplexity = min(TSNE_PERPLEXITY, max(2, len(embeddings) - 1))
    tsne = TSNE(
        n_components=2,
        perplexity=perplexity,
        learning_rate=TSNE_LEARNING_RATE,
        max_iter=TSNE_MAX_ITER,
        init="pca",
        random_state=SEED,
    )
    return tsne.fit_transform(embeddings)


def plot_tsne_map_exact(reduced_embeddings, topics, out_dir):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    if reduced_embeddings is None or len(reduced_embeddings) == 0:
        return None

    coords = np.asarray(reduced_embeddings)
    topic_values = np.asarray(topics, dtype=int)

    plt.style.use("ggplot")
    fig, ax = plt.subplots(figsize=(10, 8), dpi=150)

    vmin = int(topic_values.min()) if len(topic_values) else -1
    vmax = int(topic_values.max()) if len(topic_values) else 0
    norm = Normalize(vmin=vmin, vmax=vmax)

    sc = ax.scatter(
        coords[:, 0],
        coords[:, 1],
        c=topic_values,
        cmap="viridis",
        norm=norm,
        s=55,
        alpha=0.95,
        linewidths=0,
    )

    ax.set_title("t-SNE Visualization of Topics", fontsize=16)
    ax.set_xlabel("t-SNE Dimension 1")
    ax.set_ylabel("t-SNE Dimension 2")

    cbar = fig.colorbar(sc, ax=ax)
    cbar.set_label("Topic")

    fig.tight_layout()
    png_path = out_dir / "bertopic_documents_tsne_map_exact.png"
    pdf_path = out_dir / "bertopic_documents_tsne_map_exact.pdf"
    fig.savefig(png_path, dpi=300, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    return png_path


In [ ]:
# Load and prepare data
df_raw, detected_encoding = read_csv_safely(INPUT_CSV)
df = prepare_input(df_raw)

print("Rows after cleaning:", len(df))
print("Detected encoding:", detected_encoding)

# Quick sanity check for mojibake before modeling
sample_text = " ".join(df["text_for_bertopic"].head(20).tolist())
print("Post-clean garble score:", mojibake_score(sample_text))
display(df.head(3))


In [ ]:
def run_bertopic_9_topics(df_subset, out_dir):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    docs = df_subset["text_for_bertopic"].tolist()
    hover_docs = df_subset["hover_text"].tolist()

    topic_model, embedding_model = build_topic_model()
    embeddings = embedding_model.encode(docs, show_progress_bar=True)

    # Step 1: fit guided BERTopic
    topics, probabilities = topic_model.fit_transform(docs, embeddings)

    initial_info = topic_model.get_topic_info().copy()
    save_csv(initial_info, out_dir / "bertopic_topic_info_before_reduction.csv")

    # Step 2: reduce to exactly TARGET_MAJOR_TOPICS non-outlier topics
    topic_model.reduce_topics(docs, nr_topics=TARGET_MAJOR_TOPICS, use_ctfidf=True)
    topics = list(topic_model.topics_)
    probabilities = topic_model.probabilities_

    # Optional step: move outliers into nearest topics if you want every document assigned
    if REDUCE_OUTLIERS and any(t == -1 for t in topics):
        new_topics = topic_model.reduce_outliers(
            docs,
            topics,
            strategy=OUTLIER_STRATEGY,
            probabilities=probabilities,
            embeddings=embeddings,
        )
        topic_model.update_topics(docs, topics=new_topics)
        topics = list(new_topics)
        probabilities = topic_model.probabilities_

    topic_info = topic_model.get_topic_info().copy()
    save_csv(topic_info, out_dir / "bertopic_topic_info_after_reduction.csv")

    major_topic_ids = [int(t) for t in topic_info["Topic"].tolist() if int(t) != -1]
    major_topic_ids = sorted(major_topic_ids)

    print("Major topics after reduction:", major_topic_ids)
    print("Number of major topics:", len(major_topic_ids))

    if len(major_topic_ids) != TARGET_MAJOR_TOPICS:
        print("Warning: the fitted model did not end up with exactly", TARGET_MAJOR_TOPICS, "non-outlier topics.")
        print("This usually means the corpus is small/noisy or the clustering settings need one more adjustment.")

    topic_terms = get_topic_terms(topic_model, major_topic_ids)
    save_csv(topic_terms, out_dir / "bertopic_topic_terms.csv")

    docs_df = df_subset.copy()
    docs_df["Topic"] = topics

    if probabilities is not None:
        if isinstance(probabilities, list):
            probabilities = np.array(probabilities)
        if getattr(probabilities, "ndim", 1) == 1:
            docs_df["topic_probability"] = probabilities
        else:
            docs_df["topic_probability"] = probabilities.max(axis=1)
            proba_df = pd.DataFrame(probabilities)
            proba_df.columns = [f"topic_prob_col_{i}" for i in range(proba_df.shape[1])]
            save_csv(proba_df, out_dir / "bertopic_topic_probabilities_matrix.csv")

    save_csv(docs_df, out_dir / "bertopic_document_topics.csv")

    # Intertopic distance map for the reduced major topics
    fig_topics = topic_model.visualize_topics(topics=major_topic_ids, use_ctfidf=True)
    try_save_plotly(
        fig_topics,
        out_dir / "bertopic_intertopic_distance_map.html",
        out_dir / "bertopic_intertopic_distance_map.png",
    )

    # Hierarchical clustering dendrogram for the reduced major topics
    hierarchical_topics = topic_model.hierarchical_topics(docs, use_ctfidf=True)
    save_csv(pd.DataFrame(hierarchical_topics), out_dir / "bertopic_hierarchical_topics.csv")

    fig_hierarchy = topic_model.visualize_hierarchy(
        hierarchical_topics=hierarchical_topics,
        topics=major_topic_ids,
        use_ctfidf=True,
    )
    try_save_plotly(
        fig_hierarchy,
        out_dir / "bertopic_hierarchy_map.html",
        out_dir / "bertopic_hierarchy_map.png",
    )

    # t-SNE document map
    reduced_embeddings = tsne_2d(embeddings)
    if reduced_embeddings is not None:
        tsne_df = pd.DataFrame(reduced_embeddings, columns=["x_tsne", "y_tsne"])
        tsne_df.insert(0, "doc_id", df_subset["doc_id"].values)
        tsne_df["Topic"] = topics
        save_csv(tsne_df, out_dir / "bertopic_tsne_coordinates.csv")

        # Static matplotlib plot in the requested style
        plot_tsne_map_exact(reduced_embeddings, topics, out_dir)

    return topic_model, docs_df, topic_info


In [ ]:
# Run whole-corpus BERTopic with 9 major topics
model_9, docs_df_9, topic_info_9 = run_bertopic_9_topics(
    df_subset=df,
    out_dir=Path(OUTPUT_DIR) / "whole_corpus"
)

display(topic_info_9)
